# 04 - Optimal Commute Routing with Multi-Candidate Stations

This notebook calculates optimal door-to-door commute times using OpenTripPlanner for transit routing between multiple candidate stations for each employee and the office, selecting the combination that minimizes total commute time.

**How it works:**
- For each employee, we have 5 candidate stations from the previous notebook
- For the office, we have 5 candidate stations from the previous notebook
- We evaluate all 25 combinations (5 employee stations × 5 office stations)
- For each combination, we calculate: walk_home_to_station + transit_time + walk_station_to_office
- We select the combination with the minimal total commute time

**Formula:** walk_home_to_station + transit_time + walk_station_to_office

In [ ]:
# Import the libraries we need for routing
import requests
import pandas as pd
import time
from dotenv import load_dotenv
import os

In [ ]:
# Load environment variables and configure OpenTripPlanner server
load_dotenv()

# OpenTripPlanner server configuration
OTP_BASE_URL = "http://localhost:8080/otp/routers/default"

# Load the office candidate stations with walking times from the previous notebook
office_candidates_df = pd.read_csv('data/jj_office_candidate_stations_with_walking.csv')
print(f"Loaded {len(office_candidates_df)} office candidate stations")
print(office_candidates_df[['station_name', 'candidate_rank', 'walking_time_min']])

# Load the employee candidate stations with walking times from the previous notebook
employee_candidates_df = pd.read_csv('data/employee_candidate_stations_with_walking.csv')
print(f"\nLoaded {len(employee_candidates_df)} employee candidate stations")
print(f"Employees with candidates: {employee_candidates_df['employee_id'].nunique()}")

# Display sample data to verify
print("\nSample employee candidates:")
print(employee_candidates_df[['employee_id', 'station_name', 'candidate_rank', 'walking_time_min']].head(10))

In [ ]:
# Load the original employee data to store our final routing results
employee_df = pd.read_csv('synthetic_employees_geocoded.csv')
print(f"Loaded {len(employee_df)} employees")

# Initialize columns to store optimal routing results
employee_df['optimal_employee_station'] = None
employee_df['optimal_employee_station_id'] = None
employee_df['optimal_employee_station_rank'] = None
employee_df['optimal_office_station'] = None
employee_df['optimal_office_station_id'] = None
employee_df['optimal_office_station_rank'] = None
employee_df['walk_home_to_station_min'] = None
employee_df['transit_time_min'] = None
employee_df['walk_station_to_office_min'] = None
employee_df['total_commute_min'] = None
employee_df['otp_route_found'] = False

print("Initialized commute columns for storing optimal routing results")

In [ ]:
# Function to get transit route between two stations using OpenTripPlanner API
def get_transit_route(from_lat, from_lon, to_lat, to_lon, date, time_str):
    """Get transit route using OpenTripPlanner API"""
    url = f"{OTP_BASE_URL}/plan"
    
    params = {
        'fromPlace': f"{from_lat},{from_lon}",
        'toPlace': f"{to_lat},{to_lon}",
        'mode': 'TRANSIT,WALK',
        'date': date,
        'time': time_str
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        if 'error' in data and data['error']['id'] == 406:
            # No transit times available - this is okay, return walking only
            return data
        
        return data
        
    except Exception as e:
        print(f"Error fetching route: {e}")
        return None

In [ ]:
# Test the routing function with a sample employee candidate and office candidate
sample_employee_candidate = employee_candidates_df.iloc[0]
sample_office_candidate = office_candidates_df.iloc[0]

print(f"Testing routing with sample stations:")
print(f"Employee: {sample_employee_candidate['employee_id']}")
print(f"Employee station: {sample_employee_candidate['station_name']} (rank {sample_employee_candidate['candidate_rank']})")
print(f"Office station: {sample_office_candidate['station_name']} (rank {sample_office_candidate['candidate_rank']})")

from_lat = sample_employee_candidate['station_latitude']
from_lon = sample_employee_candidate['station_longitude']
to_lat = sample_office_candidate['station_latitude']
to_lon = sample_office_candidate['station_longitude']

print(f"From: ({from_lat}, {from_lon})")
print(f"To: ({to_lat}, {to_lon})")

# Set date for routing (within GTFS range - our GTFS data has April-December 2026)
route_date = "2026-07-08"
route_time = "08:00"

route_data = get_transit_route(from_lat, from_lon, to_lat, to_lon, route_date, route_time)

if route_data and 'plan' in route_data:
    if route_data['plan']['itineraries']:
        itinerary = route_data['plan']['itineraries'][0]
        print(f"Transit time: {itinerary['transitTime']}s ({itinerary['transitTime']/60:.1f} min)")
        print(f"Walking time: {itinerary['walkTime']}s ({itinerary['walkTime']/60:.1f} min)")
        print(f"Total time: {itinerary['duration']}s ({itinerary['duration']/60:.1f} min)")
        
        # Calculate total commute using our walking data from the previous notebook
        employee_walk = sample_employee_candidate['walking_time_min']
        office_walk = sample_office_candidate['walking_time_min']
        total_using_our_data = employee_walk + (itinerary['transitTime']/60) + office_walk
        print(f"Using our walking data: {employee_walk:.1f} + {itinerary['transitTime']/60:.1f} + {office_walk:.1f} = {total_using_our_data:.1f} min")
    else:
        print("No itineraries found")
else:
    print("Error in routing")

In [ ]:
# Set up the multi-candidate routing process
print("Starting multi-candidate optimal routing...")
print(f"Employees: {employee_df['employee_id'].nunique()}")
print(f"Employee candidates per employee: 5")
print(f"Office candidates: {len(office_candidates_df)}")
print(f"Total combinations to evaluate: {employee_df['employee_id'].nunique() * 5 * 5}")

print(f"\nUsing date: {route_date}, time: {route_time}")
print(f"Formula: walk_home_to_station + transit_time + walk_station_to_office")

In [ ]:
# Main routing loop - evaluate all station combinations for each employee
import time

employees_processed = 0
combinations_evaluated = 0
cache_file = 'data/routing_cache.csv'

# Load cache if exists to avoid recalculating the same routes
if os.path.exists(cache_file):
    print("Loading routing cache...")
    routing_cache = pd.read_csv(cache_file)
    print(f"Loaded {len(routing_cache)} cached routes")
else:
    routing_cache = pd.DataFrame(columns=['cache_key', 'employee_station_id', 'office_station_id', 'transit_time_min', 'route_found'])
    print("No cache found, starting fresh")

print("\nProcessing employees...")

# Process each employee individually
for employee_id in employee_df['employee_id']:
    print(f"\nProcessing {employee_id} ({employees_processed + 1}/{len(employee_df)})...")
    
    # Get this employee's 5 candidate stations
    emp_candidates = employee_candidates_df[employee_candidates_df['employee_id'] == employee_id]
    
    best_total_time = float('inf')
    best_route = None
    
    # Evaluate all combinations of employee station × office station (5 × 5 = 25 combinations)
    for _, emp_candidate in emp_candidates.iterrows():
        for _, office_candidate in office_candidates_df.iterrows():
            
            # Check if this combination is already cached
            cache_key = f"{emp_candidate['station_id']}_{office_candidate['station_id']}"
            if len(routing_cache) > 0 and cache_key in routing_cache['cache_key'].values:
                # Use cached transit time to save API calls
                cached_route = routing_cache[routing_cache['cache_key'] == cache_key].iloc[0]
                transit_time = cached_route['transit_time_min']
                route_found = cached_route['route_found']
            else:
                # Calculate transit time via OpenTripPlanner
                route_data = get_transit_route(
                    emp_candidate['station_latitude'], emp_candidate['station_longitude'],
                    office_candidate['station_latitude'], office_candidate['station_longitude'],
                    route_date, route_time
                )
                
                if route_data and 'plan' in route_data and route_data['plan']['itineraries']:
                    itinerary = route_data['plan']['itineraries'][0]
                    transit_time = itinerary['transitTime'] / 60
                    route_found = True
                else:
                    transit_time = None
                    route_found = False
                
                # Add to cache
                new_cache_entry = pd.DataFrame([{
                    'cache_key': cache_key,
                    'employee_station_id': emp_candidate['station_id'],
                    'office_station_id': office_candidate['station_id'],
                    'transit_time_min': transit_time,
                    'route_found': route_found
                }])
                routing_cache = pd.concat([routing_cache, new_cache_entry], ignore_index=True)
            
            # Calculate total commute time for this combination
            if route_found and transit_time is not None:
                total_time = emp_candidate['walking_time_min'] + transit_time + office_candidate['walking_time_min']
                
                # Check if this is the best combination so far
                if total_time < best_total_time:
                    best_total_time = total_time
                    best_route = {
                        'employee_station': emp_candidate['station_name'],
                        'employee_station_id': emp_candidate['station_id'],
                        'employee_station_rank': emp_candidate['candidate_rank'],
                        'office_station': office_candidate['station_name'],
                        'office_station_id': office_candidate['station_id'],
                        'office_station_rank': office_candidate['candidate_rank'],
                        'walk_home_to_station': emp_candidate['walking_time_min'],
                        'transit_time': transit_time,
                        'walk_station_to_office': office_candidate['walking_time_min'],
                        'total_commute': total_time,
                        'route_found': True
                    }
            
            combinations_evaluated += 1
    
    # Store the best route for this employee
    if best_route:
        employee_idx = employee_df[employee_df['employee_id'] == employee_id].index[0]
        employee_df.loc[employee_idx, 'optimal_employee_station'] = best_route['employee_station']
        employee_df.loc[employee_idx, 'optimal_employee_station_id'] = best_route['employee_station_id']
        employee_df.loc[employee_idx, 'optimal_employee_station_rank'] = best_route['employee_station_rank']
        employee_df.loc[employee_idx, 'optimal_office_station'] = best_route['office_station']
        employee_df.loc[employee_idx, 'optimal_office_station_id'] = best_route['office_station_id']
        employee_df.loc[employee_idx, 'optimal_office_station_rank'] = best_route['office_station_rank']
        employee_df.loc[employee_idx, 'walk_home_to_station_min'] = best_route['walk_home_to_station']
        employee_df.loc[employee_idx, 'transit_time_min'] = best_route['transit_time']
        employee_df.loc[employee_idx, 'walk_station_to_office_min'] = best_route['walk_station_to_office']
        employee_df.loc[employee_idx, 'total_commute_min'] = best_route['total_commute']
        employee_df.loc[employee_idx, 'otp_route_found'] = best_route['route_found']
        
        print(f"Best route: {best_route['employee_station']} → {best_route['office_station']}")
        print(f"Total commute: {best_route['total_commute']:.1f} min (walk: {best_route['walk_home_to_station']:.1f} + transit: {best_route['transit_time']:.1f} + walk: {best_route['walk_station_to_office']:.1f})")
    else:
        print(f"No valid route found for {employee_id}")
    
    employees_processed += 1
    
    # Save cache and progress periodically
    if employees_processed % 10 == 0:
        routing_cache.to_csv(cache_file, index=False)
        employee_df.to_csv('data/routing_progress.csv', index=False)
        print(f"Saved progress: {employees_processed} employees processed")

# Final save
routing_cache.to_csv(cache_file, index=False)
employee_df.to_csv('data/employees_with_optimal_commutes.csv', index=False)

print(f"\nRouting complete!")
print(f"Processed {employees_processed} employees")
print(f"Evaluated {combinations_evaluated} station combinations")
print(f"Saved results to data/employees_with_optimal_commutes.csv")

In [8]:
# Final cache save
routing_cache.to_csv(cache_file, index=False)
print(f"\nSaved final routing cache ({len(routing_cache)} routes)")

print(f"\nCompleted: {employees_processed} employees processed")
print(f"Combinations evaluated: {combinations_evaluated}")

# Save final results
employee_df.to_csv('data/synthetic_employees_with_optimal_commutesdata/data/.csv', index=False)
print("Saved optimal commute results to data/synthetic_employees_with_optimal_commutesdata/data/.csv")

# Display summary statistics
print("\n=== Summary Statistics ===")
print(f"Employees with optimal routes: {employee_df['otp_route_found'].sum()}")
print(f"Employees without routes: {(~employee_df['otp_route_found']).sum()}")

if employee_df['otp_route_found'].sum() > 0:
    print(f"\nTotal commute time statistics:")
    print(employee_df[employee_df['otp_route_found']]['total_commute_min'].describe())
    
    print(f"\nComponent breakdown:")
    print(f"Walking (home to station): {employee_df[employee_df['otp_route_found']]['walk_home_to_station_min'].mean():.1f} min")
    print(f"Transit time: {employee_df[employee_df['otp_route_found']]['transit_time_min'].mean():.1f} min")
    print(f"Walking (station to office): {employee_df[employee_df['otp_route_found']]['walk_station_to_office_min'].mean():.1f} min")
    
    print(f"\nSample optimal routes:")
    print(employee_df[employee_df['otp_route_found']][
        ['employee_id', 'optimal_employee_station', 'optimal_employee_station_rank',
         'optimal_office_station', 'optimal_office_station_rank',
         'walk_home_to_station_min', 'transit_time_min', 'walk_station_to_office_min', 'total_commute_min']
    ].head(10))


Saved final routing cache (1325 routes)

Completed: 178 employees processed
Combinations evaluated: 4400
Saved optimal commute results to data/synthetic_employees_with_optimal_commutes.csv

=== Summary Statistics ===
Employees with optimal routes: 131
Employees without routes: 47

Total commute time statistics:
count     131.000000
unique     50.000000
top        80.276667
freq       22.000000
Name: total_commute_min, dtype: float64

Component breakdown:
Walking (home to station): 5.6 min
Transit time: 49.5 min
Walking (station to office): 3.8 min

Sample optimal routes:
   employee_id         optimal_employee_station optimal_employee_station_rank  \
0       EMP001                     Zimmerstraße                             3   
3       EMP004                Rübke, Kurzer Weg                             1   
4       EMP005                     Zimmerstraße                             3   
5       EMP006  Eißendorfer Straße (TU Hamburg)                             5   
6       EMP007  

In [9]:


# Load office coordinates
import json
with open('data/data/jj_office_snapped.json', 'r') as f:
    office_data = json.load(f)

office_lat = office_data['snapped_latitude']
office_lon = office_data['snapped_longitude']

# Load your OpenRouteService API key
from dotenv import load_dotenv
import os
import requests
import time

load_dotenv()
ORS_API_KEY = os.getenv('OPENROUTESERVICE_API_KEY')

# Parking time overhead (time to find parking + walk from parking to office)
PARKING_TIME_MIN = 8  # Average 8 minutes for parking search + walking

def get_car_route(origin_lat, origin_lon, dest_lat, dest_lon, api_key, max_retries=3):
    """Get car driving route using OpenRouteService"""
    url = "https://api.openrouteservice.org/v2/directions/driving-car"
    
    headers = {
        'Authorization': api_key,
        'Content-Type': 'application/json',
        'Accept': 'application/json'
    }
    
    body = {
        'coordinates': [[origin_lon, origin_lat], [dest_lon, dest_lat]]
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.post(url, json=body, headers=headers)
            
            if response.status_code == 429:
                wait_time = 2 ** attempt
                print(f"Rate limited, waiting {wait_time}s...")
                time.sleep(wait_time)
                continue
                
            if response.status_code == 403:
                print(f"API quota exceeded, waiting 30s...")
                time.sleep(30)
                continue
                
            response.raise_for_status()
            data = response.json()
            
            if 'routes' in data and len(data['routes']) > 0:
                route = data['routes'][0]
                distance_m = route['summary']['distance']
                duration_s = route['summary']['duration']
                return distance_m, duration_s
            else:
                return None, None
                
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"Error after {max_retries} attempts: {e}")
                return None, None
            time.sleep(2)
    
    return None, None



In [10]:
# Initialize car routing columns in employee_df
employee_df['car_distance_m'] = None
employee_df['car_driving_time_min'] = None
employee_df['car_parking_time_min'] = PARKING_TIME_MIN
employee_df['car_total_time_min'] = None
 
cache_file = 'data/car_routing_cache.csv'
 
# Load cache if exists
if os.path.exists(cache_file):
    print("Loading cached car routes...")
    try:
        car_cache = pd.read_csv(cache_file)
        
        # Check if cache has the expected columns
        if 'car_driving_time_min' in car_cache.columns:
            for idx, row in car_cache.iterrows():
                mask = employee_df['employee_id'] == row['employee_id']
                employee_df.loc[mask, 'car_distance_m'] = row['car_distance_m']
                employee_df.loc[mask, 'car_driving_time_min'] = row['car_driving_time_min']
                employee_df.loc[mask, 'car_total_time_min'] = row['car_driving_time_min'] + PARKING_TIME_MIN
            
            already_calculated = employee_df['car_distance_m'].notna().sum()
            print(f"Loaded {already_calculated} cached car routes")
        else:
            print("Cache has different structure, recalculating...")
            car_cache = pd.DataFrame()
            
    except Exception as e:
        print(f"Error loading cache: {e}, starting fresh")
        car_cache = pd.DataFrame()
else:
    car_cache = pd.DataFrame()
 
# Calculate missing car routes
to_calculate = employee_df[employee_df['car_distance_m'].isna()]
 
print(f"Need to calculate car routes for {len(to_calculate)} employees")

Loading cached car routes...
Loaded 178 cached car routes
Need to calculate car routes for 0 employees


In [11]:
processed_count = 0
failed_count = 0
 
for idx, row in to_calculate.iterrows():
    print(f"Processing {row['employee_id']} ({processed_count + 1}/{len(to_calculate)})...")
    
    try:
        dist_m, time_s = get_car_route(
            row['snapped_latitude'], row['snapped_longitude'],
            office_lat, office_lon,
            ORS_API_KEY
        )
        
        if dist_m is not None and time_s is not None:
            driving_time_min = time_s / 60
            total_time_min = driving_time_min + PARKING_TIME_MIN
            
            employee_df.loc[idx, 'car_distance_m'] = dist_m
            employee_df.loc[idx, 'car_driving_time_min'] = driving_time_min
            employee_df.loc[idx, 'car_total_time_min'] = total_time_min
            
            print(f"  Distance: {dist_m/1000:.1f}km, Driving: {driving_time_min:.1f}min, Total (with parking): {total_time_min:.1f}min")
            processed_count += 1
        else:
            print(f"  Failed to get car route")
            failed_count += 1
            
    except Exception as e:
        print(f"  Error: {e}")
        failed_count += 1
    
    time.sleep(0.5)
    
    if processed_count % 10 == 0:
        car_cache = employee_df[['employee_id', 'car_distance_m', 'car_driving_time_min']].dropna()
        car_cache.to_csv(cache_file, index=False)
        print(f"Saved progress: {processed_count} calculated")
 


In [12]:
# Final save
car_cache = employee_df[['employee_id', 'car_distance_m', 'car_driving_time_min']].dropna()
car_cache.to_csv(cache_file, index=False)
print(f"\nCompleted: {processed_count} calculated, {failed_count} failed")

# Save updated results with car routing data
employee_df.to_csv('data/synthetic_employees_with_optimal_commutesdata/data/.csv', index=False)
print("Updated results with car routing data")

print(f"\nCar routing summary:")
print(f"Employees with car routes: {employee_df['car_distance_m'].notna().sum()}")
print(f"Average car distance: {employee_df['car_distance_m'].mean()/1000:.1f} km")
print(f"Average car driving time: {employee_df['car_driving_time_min'].mean():.1f} min")
print(f"Average car total time (with {PARKING_TIME_MIN}min parking): {employee_df['car_total_time_min'].mean():.1f} min")


Completed: 0 calculated, 0 failed
Updated results with car routing data

Car routing summary:
Employees with car routes: 178
Average car distance: 24.4 km
Average car driving time: 38.8 min
Average car total time (with 8min parking): 46.8 min
